# Notebook 04 — Warehouse Optimization
## Best Case vs. Worst Case

| | Best Design | Worst Design |
|---|---|---|
| **Sizing** | Right-sized (no spilling) | XS for complex aggregation (spills to remote) |
| **Concurrency** | Multi-cluster auto-scale | Single cluster, queries queued |
| **QAS** | Enabled for ad-hoc fraud scans | Disabled — slow serial execution |

In [ ]:
USE SCHEMA JPMC_PERF_HOL.CONSUMER_BANKING;

---
## Step 1: Warehouse Sizing — Spilling vs. No Spilling

**Scenario**: Monthly fraud loss report — aggregates 500M rows by region and merchant category.

### Worst Case: XS Warehouse (spills to disk)

In [ ]:
USE WAREHOUSE HOL_XS;

-- Complex aggregation on XS — will spill
SELECT
    a.account_type,
    t.channel,
    DATE_TRUNC('month', t.transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(t.amount) AS total_amount,
    SUM(CASE WHEN t.fraud_flag THEN t.amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS t
JOIN ACCOUNTS a ON t.account_id = a.account_id
WHERE t.transaction_date >= '2024-01-01'
GROUP BY 1, 2, 3
ORDER BY fraud_losses DESC;

**Check the Query Profile** → Look for:
- `Bytes spilled to local storage` (bad)
- `Bytes spilled to remote storage` (worse!)
- Execution time

### Best Case: Medium Warehouse (no spilling)

In [ ]:
USE WAREHOUSE HOL_M;

-- Same query on Medium — fits in memory
SELECT
    a.account_type,
    t.channel,
    DATE_TRUNC('month', t.transaction_date) AS txn_month,
    COUNT(*) AS txn_count,
    SUM(t.amount) AS total_amount,
    SUM(CASE WHEN t.fraud_flag THEN t.amount ELSE 0 END) AS fraud_losses
FROM TRANSACTIONS t
JOIN ACCOUNTS a ON t.account_id = a.account_id
WHERE t.transaction_date >= '2024-01-01'
GROUP BY 1, 2, 3
ORDER BY fraud_losses DESC;

**Check the Query Profile** → Compare:
- Zero spilling
- Significantly faster execution
- Credits used: 4x more per hour BUT query finishes 5-10x faster → net cheaper

---
## Step 2: Compare Metrics

In [ ]:
SELECT
    warehouse_name,
    warehouse_size,
    total_elapsed_time / 1000 AS elapsed_sec,
    bytes_spilled_to_local_storage / (1024*1024*1024) AS spill_local_gb,
    bytes_spilled_to_remote_storage / (1024*1024*1024) AS spill_remote_gb,
    partitions_scanned
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_HISTORY
WHERE start_time > DATEADD(minute, -15, CURRENT_TIMESTAMP())
  AND query_text ILIKE '%fraud_losses%'
ORDER BY start_time DESC
LIMIT 2;

---
## Step 3: Query Acceleration Service (QAS)

**Scenario**: Fraud analyst runs ad-hoc scan — "Find all transactions over $5,000 in the last week grouped by merchant."

### Without QAS

In [ ]:
-- Create a warehouse WITHOUT QAS
CREATE WAREHOUSE IF NOT EXISTS HOL_NO_QAS
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    ENABLE_QUERY_ACCELERATION = FALSE;

USE WAREHOUSE HOL_NO_QAS;

SELECT
    merchant_name,
    COUNT(*) AS high_value_txns,
    SUM(amount) AS total_amount
FROM TRANSACTIONS
WHERE amount > 500
  AND transaction_date >= '2024-11-01'
GROUP BY 1
ORDER BY total_amount DESC
LIMIT 20;

### With QAS Enabled

In [ ]:
-- Create a warehouse WITH QAS
CREATE WAREHOUSE IF NOT EXISTS HOL_QAS
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    ENABLE_QUERY_ACCELERATION = TRUE
    QUERY_ACCELERATION_MAX_SCALE_FACTOR = 8;

USE WAREHOUSE HOL_QAS;

SELECT
    merchant_name,
    COUNT(*) AS high_value_txns,
    SUM(amount) AS total_amount
FROM TRANSACTIONS
WHERE amount > 500
  AND transaction_date >= '2024-11-01'
GROUP BY 1
ORDER BY total_amount DESC
LIMIT 20;

**Check Query Profile** → Look for "Query Acceleration" section showing partitions offloaded to the service.

---
## Step 4: Identify QAS-Eligible Queries

In [ ]:
-- Which queries would benefit most from QAS?
SELECT
    query_id,
    SUBSTR(query_text, 1, 80) AS query_preview,
    eligible_query_acceleration_time / 1000 AS eligible_time_sec
FROM SNOWFLAKE.ACCOUNT_USAGE.QUERY_ACCELERATION_ELIGIBLE
WHERE start_time > DATEADD(hour, -1, CURRENT_TIMESTAMP())
ORDER BY eligible_query_acceleration_time DESC
LIMIT 5;

---
## Key Takeaways

| | Best Design | Worst Design |
|---|---|---|
| **Sizing** | Match warehouse to query complexity (M for JOINs/aggs) | XS for everything → spilling = 10x slower |
| **Over-sizing** | — | 4XL for simple lookups → same speed, 16x the cost |
| **QAS** | Enable for ad-hoc fraud/analytics warehouses | Disabled → slow scans for eligible queries |
| **Multi-cluster** | Auto-scale for concurrent branch reporting | Single cluster → queuing during peak hours |

**Rule of thumb**: If a query spills, size up. If it doesn't spill on XS, don't size up.

**Next** → [Notebook 05 — Search Optimization Service](./Notebook_05_Search_Optimization.ipynb)

---
## Cleanup

In [ ]:
USE WAREHOUSE HOL_M;
-- DROP WAREHOUSE IF EXISTS HOL_NO_QAS;
-- DROP WAREHOUSE IF EXISTS HOL_QAS;